# TRELLIS バックエンド on Google Colab

**目的**: EC3D-Bridge のローカル backend から Colab の T4 GPU で動く TRELLIS を呼べるようにする。
完全無料で月 50 個以上の 3D 生成が可能 (Colab T4 無料枠 = 週 12〜15時間)。

**使い方**:
1. ランタイム → ランタイムのタイプを変更 → **T4 GPU** を選ぶ
2. 上から順にセルを実行 (`Shift+Enter`)
3. 最後のセルの出力に `Backend URL: https://xxx.ngrok-free.app` と出る
4. その URL を EC3D-Bridge の `.env` に貼る:
   ```
   MODEL_PROVIDER=colab_trellis
   TRELLIS_COLAB_URL=https://xxx.ngrok-free.app
   ```
5. ローカルで `uvicorn main:app` を起動して通常通り使う

**制約**: Colab は 12 時間でセッションが切れます。切れたらこのノートブックを開き直して全セル実行 → 新しい URL を `.env` に反映。

## 1. 依存をインストール

In [ ]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio python-multipart
!pip install -q torch torchvision
# TRELLIS 本体 (Microsoft の公式リポジトリ)
!git clone -q https://github.com/microsoft/TRELLIS.git || true
%cd TRELLIS
!pip install -q -r requirements.txt
!pip install -q -e .

## 2. TRELLIS パイプラインを GPU にロード

In [ ]:
import os, io, tempfile
import torch
from PIL import Image

os.environ['ATTN_BACKEND'] = 'xformers'
os.environ['SPCONV_ALGO'] = 'native'

from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils

print('Loading TRELLIS-image-large on GPU...')
pipeline = TrellisImageTo3DPipeline.from_pretrained('microsoft/TRELLIS-image-large')
pipeline.cuda()
print('✓ TRELLIS ロード完了')

## 3. 1 枚生成する関数 (テスト用)

In [ ]:
def generate_glb_from_image(image_bytes: bytes) -> bytes:
    image = Image.open(io.BytesIO(image_bytes)).convert('RGB')
    outputs = pipeline.run(
        image,
        seed=42,
        # 推奨デフォルト。品質と速度のバランス。
        sparse_structure_sampler_params={'steps': 20, 'cfg_strength': 7.5},
        slat_sampler_params={'steps': 20, 'cfg_strength': 3.0},
    )
    glb = postprocessing_utils.to_glb(
        outputs['gaussian'][0],
        outputs['mesh'][0],
        simplify=0.95,
        texture_size=1024,
    )
    with tempfile.NamedTemporaryFile(suffix='.glb', delete=False) as f:
        glb.export(f.name)
        with open(f.name, 'rb') as fh:
            return fh.read()

## 4. FastAPI バックエンドを起動 + ngrok でトンネル

In [ ]:
from fastapi import FastAPI, UploadFile, File, Response, HTTPException
from pyngrok import ngrok
import uvicorn, nest_asyncio, threading

app = FastAPI(title='TRELLIS Colab Backend')

@app.get('/health')
def health():
    return {'status': 'ok', 'cuda_available': torch.cuda.is_available()}

@app.post('/generate')
async def generate(file: UploadFile = File(...)):
    img_bytes = await file.read()
    if len(img_bytes) == 0:
        raise HTTPException(400, '空ファイル')
    try:
        glb = generate_glb_from_image(img_bytes)
    except Exception as e:
        raise HTTPException(500, f'TRELLIS error: {e}')
    return Response(content=glb, media_type='model/gltf-binary')

# ngrok トンネル
# 初回は ngrok の auth token を設定する必要あり (無料登録で取れる)
# https://dashboard.ngrok.com/get-started/your-authtoken からコピー
# 以下の行のコメントを外してトークンを貼ってください:
# ngrok.set_auth_token('YOUR_NGROK_AUTHTOKEN_HERE')

public_url = ngrok.connect(8000).public_url
print('=' * 60)
print(f'🎉 Backend URL: {public_url}')
print('   ↑ これを EC3D-Bridge の .env に貼ってください:')
print(f'   TRELLIS_COLAB_URL={public_url}')
print(f'   MODEL_PROVIDER=colab_trellis')
print('=' * 60)

nest_asyncio.apply()
uvicorn.run(app, host='0.0.0.0', port=8000)